# Support-point CNN supervised training — Kaggle CUDA-BP notebook

This notebook is prepared for the `cnn-support-point-estimator` branch. It:

1. clones the current branch into `/kaggle/working`,
2. samples support-point configurations in the 7×7 lattice (`0..48`),
3. labels them with the **exact CUDA bit-parallel distance kernel** (`kernel_cuda_bp.py`),
4. trains a heuristic support-point estimator (`support_point_cnn_estimator.py`) — by default a small **PyTorch CNN on CUDA** in Kaggle, with a CPU NumPy ridge fallback, and
5. reports train/validation MSE/MAE/RMSE losses and **accuracy = percentage of predictions within ±3 of the actual minimum distance**.

Important separation:
- **Exact CUDA labeling** below is the supervised-data/certification step. It uses `DistanceOracleCUDABP.max_zeros_batch(..., sample_count=0)` and records `min_distance = 49 - max_zeros`.
- **CNN inference** is heuristic only. It is useful for ranking or triage, but it does **not** certify distances. Re-check selected supports with the CUDA kernel before treating a value as exact.

## Practical Kaggle setup notes

- In Kaggle, choose **Settings → Accelerator → GPU**. T4/P100/A100 are all fine; exact labeling is much faster on A100/T4 than CPU fallback workflows.
- The default training backend is now `TRAINING_BACKEND = "torch"`, which uses PyTorch and selects CUDA when available. Set `TRAINING_BACKEND = "ridge"` to reproduce the older NumPy feature-extractor + ridge path; that path is CPU-only by design.
- If you saw near-zero GPU utilization in earlier runs, that was expected for the training cell: it used NumPy + `np.linalg.solve` on CPU. The GPU was only used during the exact CUDA-BP label/re-check cells, which are very short for `k <= 8` and may appear as brief spikes in `nvidia-smi`.
- Even with the torch backend, this is a tiny 7×7 model over only a few thousand examples, so GPU utilization may be bursty rather than a sustained 90%+ workload. The important check is that the training cell prints `Torch CNN training device: cuda...`.
- Turn **Internet on** for the clone cell, or attach this repository as a Kaggle Dataset and replace the clone cell with a local copy command.
- The branch named below must be pushed to GitHub before running this notebook on Kaggle.
- Kaggle usually includes CuPy and PyTorch for NVIDIA GPUs. If the CUDA import cell fails, restart with a GPU accelerator and install matching packages only if necessary.
- The default configuration below is intended as a **real supervised-data run**, not a smoke test: 2,048 exactly labeled supports over `k=3..8`, balanced by support size. On slower/free GPUs this can take a while; for a quick smoke run use `N_SAMPLES = 256..512` and `K_MAX = 7`.
- Exact labeling is exponential in support size: exact CUDA-BP supports `k <= 21`, but practical supervised dataset generation should increase `N_SAMPLES`, `K_MAX`, and `LABEL_BATCH_SIZE` gradually after timing your GPU.
- Artifacts are written under `/kaggle/working/support_point_cnn_runs/<run-name>` so they can be downloaded from the Kaggle output panel.

In [ ]:
# Clone the current branch.
# If you forked the repo, update REPO_URL. If the branch name changed, update BRANCH.
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/t-ravnushkin/generalized-toric-gpu-search.git"
BRANCH = "cnn-support-point-estimator"
REPO_DIR = Path("/kaggle/working/generalized-toric-gpu-search")

if REPO_DIR.exists():
    print(f"Removing existing checkout: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

cmd = ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Checked out:")
subprocess.run(["git", "status", "--short", "--branch"], check=True)

In [ ]:
# CUDA and project imports. This notebook requires CUDA for exact labeling.
import importlib
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

import cupy as cp
n_gpu = cp.cuda.runtime.getDeviceCount()
if n_gpu <= 0:
    raise RuntimeError("No CUDA GPU found. In Kaggle, enable Settings → Accelerator → GPU and restart.")

print(f"CuPy {cp.__version__}; CUDA devices: {n_gpu}")
for i in range(n_gpu):
    props = cp.cuda.runtime.getDeviceProperties(i)
    name = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    print(f"  device {i}: {name}, {props.get('multiProcessorCount', '?')} SMs, {props['totalGlobalMem'] / 1024**3:.1f} GiB")

try:
    import torch
    TORCH_AVAILABLE = True
    print(f"PyTorch {torch.__version__}; torch.cuda.is_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  torch cuda:{i}: {torch.cuda.get_device_name(i)}")
except ImportError:
    torch = None
    TORCH_AVAILABLE = False
    print("PyTorch is not installed; set TRAINING_BACKEND='ridge' or install torch for GPU CNN training.")

from support_point_cnn_estimator import (
    GRID,
    N_POINTS,
    Example,
    RidgeCnnEstimator,
    TorchCnnEstimator,
    one_point_extensions,
    train_estimator,
    train_torch_cnn_estimator,
    train_validation_split,
)

# Avoid importing precompute.py here: it imports pyopencl at module import time. For this CUDA-only notebook,
# build the same evaluation matrix directly from gf8.py.
from gf8 import gf8_mul, gf8_pow
from kernel_cuda_bp import init_cuda_oracles_bp

print(f"Grid={GRID}x{GRID}, N_POINTS={N_POINTS}")

In [ ]:
# Configuration for a larger CUDA-labeled supervised dataset run.
# Defaults are meant to be uploaded to Kaggle and run directly on a GPU notebook.
# For a quick smoke test, set N_SAMPLES=256 or 512 and K_MAX=7.
SEED = 20260608
N_SAMPLES = 2048          # serious default: 4x the earlier 512-row diagnostic run
K_MIN = 3                 # k=2 is essentially all distance 42, so spend labels on nontrivial supports
K_MAX = 8                 # k=8 is substantially costlier than k=7; raise only after timing your GPU
LABEL_BATCH_SIZE = 128    # supports per CUDA launch group, grouped by k; lower if GPU memory is tight
VAL_FRACTION = 0.20
RIDGE_GRID = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0]
BALANCE_BY = "k-target"   # helps reduce per-k/label-prior collapse on imbalanced exact labels
ACCURACY_TOLERANCE = 3.0

# "torch" trains a learnable 7x7 support-mask CNN with PyTorch/CUDA on Kaggle.
# "ridge" reproduces the older NumPy fixed-feature + ridge regression backend and is CPU-only.
TRAINING_BACKEND = "torch"
TORCH_DEVICE = "auto"     # "auto" => cuda if torch sees a GPU, otherwise cpu
TORCH_EPOCHS = 500
TORCH_BATCH_SIZE = 256
TORCH_LR = 1e-3
TORCH_WEIGHT_DECAY = 1e-4
TORCH_PATIENCE = 75

RUN_NAME = f"n{N_SAMPLES}_k{K_MIN}-{K_MAX}_seed{SEED}_cuda_bp_exact_{TRAINING_BACKEND}"
RUN_DIR = Path("/kaggle/working/support_point_cnn_runs") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATASET_JSONL = RUN_DIR / "support_point_cuda_bp_labeled.jsonl"
MODEL_NPZ = RUN_DIR / "support_point_cnn_ridge_model.npz"
MODEL_PT = RUN_DIR / "support_point_torch_cnn_model.pt"
PREDICTIONS_CSV = RUN_DIR / "support_point_cnn_predictions.csv"

print({
    "SEED": SEED,
    "N_SAMPLES": N_SAMPLES,
    "K_MIN": K_MIN,
    "K_MAX": K_MAX,
    "LABEL_BATCH_SIZE": LABEL_BATCH_SIZE,
    "VAL_FRACTION": VAL_FRACTION,
    "RIDGE_GRID": RIDGE_GRID,
    "BALANCE_BY": BALANCE_BY,
    "TRAINING_BACKEND": TRAINING_BACKEND,
    "TORCH_DEVICE": TORCH_DEVICE,
    "TORCH_EPOCHS": TORCH_EPOCHS,
    "TORCH_BATCH_SIZE": TORCH_BATCH_SIZE,
    "RUN_DIR": str(RUN_DIR),
})

In [ ]:
# Support sampling utilities. These only choose candidate supports; no labels are created here.
def sample_supports(n: int, k_min: int, k_max: int, seed: int) -> list[tuple[int, ...]]:
    if n <= 0:
        raise ValueError("n must be positive")
    if not (1 <= k_min <= k_max <= N_POINTS):
        raise ValueError(f"expected 1 <= k_min <= k_max <= {N_POINTS}")
    rng = random.Random(seed)
    supports: list[tuple[int, ...]] = []
    seen: set[tuple[int, ...]] = set()
    ks = list(range(k_min, k_max + 1))
    attempts = 0
    while len(supports) < n:
        attempts += 1
        if attempts > max(1000, n * 500):
            raise RuntimeError("could not generate enough unique supports; lower n or widen k range")
        k = ks[len(supports) % len(ks)]  # roughly stratified by k
        support = tuple(sorted(rng.sample(range(N_POINTS), k)))
        if support in seen:
            continue
        seen.add(support)
        supports.append(support)
    random.Random(seed + 1).shuffle(supports)
    return supports

def build_eval_matrix_cuda_input() -> np.ndarray:
    lattice = [(a, b) for a in range(GRID) for b in range(GRID)]
    torus = [(x, y) for x in range(1, 8) for y in range(1, 8)]
    M = np.zeros((N_POINTS, N_POINTS), dtype=np.int32)
    for i, (a, b) in enumerate(lattice):
        for j, (x, y) in enumerate(torus):
            M[i, j] = gf8_mul(gf8_pow(x, a), gf8_pow(y, b))
    return M

supports = sample_supports(N_SAMPLES, K_MIN, K_MAX, SEED)
print(f"Sampled {len(supports)} supports")
print(pd.Series([len(s) for s in supports]).value_counts().sort_index().rename("count_by_k"))
print("First 5 supports:", supports[:5])

## Exact CUDA-BP labeling step

This is the only step that creates ground-truth labels. It uses the repository CUDA bit-parallel kernel with `sample_count=0` and `target_distance=1` so each supported coefficient space is fully scanned for `k <= 21`.

The CNN is not involved in this cell.

In [ ]:
def label_supports_cuda_bp(
    supports: list[tuple[int, ...]],
    *,
    label_batch_size: int = 256,
    dataset_path: Path | None = None,
) -> list[dict]:
    if any(len(s) > 21 for s in supports):
        too_large = sorted({len(s) for s in supports if len(s) > 21})
        raise ValueError(f"Exact CUDA-BP labeling supports k <= 21; got k values {too_large}")

    M = build_eval_matrix_cuda_input()
    oracles, sm_count = init_cuda_oracles_bp(M)
    oracle = oracles[0]  # Kaggle usually has one GPU; the batch kernel already parallelizes across supports.
    print(f"CUDA-BP exact labeler ready on device 0; sm_count={sm_count}")

    grouped: dict[int, list[tuple[int, tuple[int, ...]]]] = defaultdict(list)
    for pos, support in enumerate(supports):
        grouped[len(support)].append((pos, support))

    rows_by_pos: dict[int, dict] = {}
    t0 = time.perf_counter()
    for k, items in sorted(grouped.items()):
        print(f"Labeling k={k}: {len(items)} supports")
        for start in range(0, len(items), label_batch_size):
            chunk = items[start:start + label_batch_size]
            batch = [list(support) for _, support in chunk]
            # target_distance=1 => tmax_zeros=48. Except for degenerate distance-0 supports,
            # this forces a full exact scan; sample_count=0 disables sampling.
            max_zeros = oracle.max_zeros_batch(batch, target_distance=1, sample_count=0)
            for (pos, support), mz in zip(chunk, max_zeros, strict=True):
                mz = int(mz)
                rows_by_pos[pos] = {
                    "indices": list(support),
                    "k": len(support),
                    "max_zeros": mz,
                    "min_distance": int(N_POINTS - mz),
                    "label_backend": "cuda-bp-exact",
                    "sample_count": 0,
                }
        cp.cuda.Device(0).synchronize()
        elapsed = time.perf_counter() - t0
        print(f"  done k={k}; elapsed={elapsed:.1f}s")

    rows = [rows_by_pos[i] for i in range(len(supports))]
    if dataset_path is not None:
        with dataset_path.open("w") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")
        print(f"Wrote exact labeled dataset: {dataset_path}")
    return rows

labeled_rows = label_supports_cuda_bp(supports, label_batch_size=LABEL_BATCH_SIZE, dataset_path=DATASET_JSONL)
labels_df = pd.DataFrame(labeled_rows)
print(labels_df.groupby("k")["min_distance"].agg(["count", "min", "mean", "max"]))
labels_df.head()

## Heuristic CNN training/inference step

From here on, the exact labels are treated as a supervised dataset. The default estimator is now a small learnable PyTorch CNN over the 7×7 support mask trained with mini-batches on CUDA (`TRAINING_BACKEND = "torch"`). The older fixed-feature NumPy/ridge estimator is still available with `TRAINING_BACKEND = "ridge"`, but that path is CPU-only.

The split is stratified by `(k, min_distance)` where possible. The copied 256-row run showed why this matters: labels were very discrete and dominated by support size, so an unstratified split plus low-capacity features can look like a near-constant/per-k-mean predictor. The cell below prints label/split diagnostics and either trains the torch CNN with early stopping or selects ridge strength from `RIDGE_GRID` using validation MAE.

The losses and ±3 accuracy below measure how well the heuristic reproduces CUDA labels on this sampled dataset; they are not distance certificates.

In [ ]:
def examples_from_rows(rows: list[dict]) -> list[Example]:
    return [Example(tuple(int(i) for i in row["indices"]), float(row["min_distance"])) for row in rows]

def regression_metrics(estimator, examples: list[Example], tolerance: float = 3.0) -> dict:
    if not examples:
        return {"count": 0, "mae": np.nan, "rmse": np.nan, "accuracy_within_pm3": np.nan, "pred_std": np.nan}
    y = np.asarray([ex.target for ex in examples], dtype=np.float64)
    pred = estimator.predict_indices([ex.indices for ex in examples])
    err = pred - y
    mse = float(np.mean(err * err))
    return {
        "count": len(examples),
        "mse_loss": mse,
        "mae_loss": float(np.mean(np.abs(err))),
        "rmse_loss": float(np.sqrt(mse)),
        "bias": float(np.mean(err)),
        "actual_std": float(np.std(y)),
        "pred_std": float(np.std(pred)),
        "accuracy_within_pm3": float(np.mean(np.abs(err) <= tolerance) * 100.0),
    }

def label_diagnostics(name: str, examples: list[Example]) -> pd.DataFrame:
    df = pd.DataFrame({"k": [len(ex.indices) for ex in examples], "min_distance": [ex.target for ex in examples]})
    print(f"\n{name}: n={len(df)}")
    print("label counts:")
    print(df["min_distance"].value_counts().sort_index())
    print("by k:")
    summary = df.groupby("k")["min_distance"].agg(["count", "min", "mean", "max", "std"]).fillna(0)
    print(summary)
    return df

examples = examples_from_rows(labeled_rows)
train_examples, val_examples = train_validation_split(examples, VAL_FRACTION, SEED, stratify=True)
all_df = label_diagnostics("all labels", examples)
train_df = label_diagnostics("train split", train_examples)
val_df = label_diagnostics("validation split", val_examples)

if TRAINING_BACKEND == "torch":
    if not TORCH_AVAILABLE:
        raise RuntimeError("TRAINING_BACKEND='torch' but PyTorch is not installed. Kaggle GPU runtimes normally include torch.")
    if TORCH_DEVICE == "auto" and not torch.cuda.is_available():
        raise RuntimeError("TRAINING_BACKEND='torch' requested, but torch cannot see CUDA. Enable Kaggle GPU or set TRAINING_BACKEND='ridge'.")
    estimator, torch_history = train_torch_cnn_estimator(
        train_examples,
        val_examples=val_examples,
        balance_by=BALANCE_BY,
        epochs=TORCH_EPOCHS,
        batch_size=TORCH_BATCH_SIZE,
        lr=TORCH_LR,
        weight_decay=TORCH_WEIGHT_DECAY,
        patience=TORCH_PATIENCE,
        seed=SEED,
        device=TORCH_DEVICE,
        verbose=True,
    )
    model_path = MODEL_PT
    estimator.save(model_path)
    history_df = pd.DataFrame(torch_history)
    print("\nTorch training history tail:")
    display(history_df.tail(10))
    if not history_df.empty:
        best_row = history_df.loc[history_df["validation_mae"].idxmin()].to_dict()
        selected_backend = f"torch-cnn:{estimator.device}"
        selected_detail = f"best_epoch={int(best_row['epoch'])} validation_MAE={best_row['validation_mae']:.3f}"
    else:
        selected_backend = f"torch-cnn:{estimator.device}"
        selected_detail = "no history rows"
elif TRAINING_BACKEND == "ridge":
    ridge_rows = []
    best = None
    for ridge in RIDGE_GRID:
        candidate = train_estimator(train_examples, ridge=float(ridge), balance_by=BALANCE_BY)
        train_m = regression_metrics(candidate, train_examples, ACCURACY_TOLERANCE)
        val_m = regression_metrics(candidate, val_examples, ACCURACY_TOLERANCE)
        row = {"ridge": float(ridge), "train_mae": train_m["mae_loss"], "validation_mae": val_m["mae_loss"],
               "train_rmse": train_m["rmse_loss"], "validation_rmse": val_m["rmse_loss"],
               "validation_pred_std": val_m["pred_std"]}
        ridge_rows.append(row)
        if best is None or row["validation_mae"] < best[0]["validation_mae"]:
            best = (row, candidate)

    ridge_df = pd.DataFrame(ridge_rows)
    print("\nRidge sweep:")
    display(ridge_df)
    best_row, estimator = best
    model_path = MODEL_NPZ
    estimator.save(model_path)
    selected_backend = "numpy-ridge:cpu"
    selected_detail = f"ridge={best_row['ridge']} validation_MAE={best_row['validation_mae']:.3f}"
else:
    raise ValueError(f"unknown TRAINING_BACKEND={TRAINING_BACKEND!r}")

metrics_df = pd.DataFrame([
    {"split": "train", "backend": selected_backend, **regression_metrics(estimator, train_examples, ACCURACY_TOLERANCE)},
    {"split": "validation", "backend": selected_backend, **regression_metrics(estimator, val_examples, ACCURACY_TOLERANCE)},
])
print(f"Selected backend: {selected_backend}; {selected_detail}")
print(f"Saved model: {model_path}")
metrics_df

In [ ]:
# Per-example predictions for inspection/download.
def prediction_frame(split_name: str, examples: list[Example]) -> pd.DataFrame:
    pred = estimator.predict_indices([ex.indices for ex in examples])
    actual = np.asarray([ex.target for ex in examples], dtype=np.float64)
    return pd.DataFrame({
        "split": split_name,
        "indices": [list(ex.indices) for ex in examples],
        "k": [len(ex.indices) for ex in examples],
        "actual_min_distance_cuda_bp": actual.astype(int),
        "predicted_min_distance_cnn": pred,
        "error_pred_minus_actual": pred - actual,
        "within_pm3": np.abs(pred - actual) <= ACCURACY_TOLERANCE,
    })

pred_df = pd.concat([
    prediction_frame("train", train_examples),
    prediction_frame("validation", val_examples),
], ignore_index=True)
pred_df.to_csv(PREDICTIONS_CSV, index=False)
print(f"Wrote predictions: {PREDICTIONS_CSV}")
print(pred_df.groupby(["split", "k"])["within_pm3"].agg(count="count", accuracy_pct=lambda s: 100 * s.mean()).round(2))
pred_df.head(10)

In [ ]:
# Optional diagnostic plot.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
colors = {"train": "tab:blue", "validation": "tab:orange"}
for split, part in pred_df.groupby("split"):
    ax.scatter(
        part["actual_min_distance_cuda_bp"],
        part["predicted_min_distance_cnn"],
        label=split,
        alpha=0.75,
        s=28,
        color=colors.get(split),
    )
lo = min(pred_df["actual_min_distance_cuda_bp"].min(), pred_df["predicted_min_distance_cnn"].min()) - 1
hi = max(pred_df["actual_min_distance_cuda_bp"].max(), pred_df["predicted_min_distance_cnn"].max()) + 1
ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, label="perfect")
ax.fill_between([lo, hi], [lo - 3, hi - 3], [lo + 3, hi + 3], color="gray", alpha=0.15, label="±3")
ax.set_xlabel("Actual min distance (CUDA-BP exact label)")
ax.set_ylabel("Predicted min distance (CNN heuristic)")
ax.set_title("Support-point CNN predictions vs exact CUDA labels")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## Heuristic-only ranking example

This cell uses only the trained CNN estimator to rank candidate one-point extensions. These numbers are **predictions**, not exact labels. Use the optional re-check cell after this if you want exact CUDA distances for selected candidates.

In [ ]:
# Choose a base support and rank all one-point extensions by predicted min distance.
BASE_SUPPORT = (0, 1, 7)
TOP_N = 25

candidates = one_point_extensions(BASE_SUPPORT)
scores = estimator.predict_indices(candidates)
order = np.argsort(scores)[::-1][:TOP_N]
ranked = pd.DataFrame([
    {
        "rank": rank,
        "added_point": sorted(set(candidates[int(pos)]) - set(BASE_SUPPORT))[0],
        "support": list(candidates[int(pos)]),
        "predicted_min_distance_cnn": float(scores[int(pos)]),
        "inference_type": "cnn-heuristic-not-certified",
    }
    for rank, pos in enumerate(order, start=1)
])
ranked

## Optional exact CUDA re-check of heuristic top candidates

Run this only when you want to convert heuristic rankings into exact labels for the selected candidates. This is a separate exact-labeling call and can be appended to your supervised dataset.

In [ ]:
# Exact CUDA labels for the heuristic top-N candidates. Keep this separate from CNN inference.
RECHECK_TOP_CANDIDATES = True

if RECHECK_TOP_CANDIDATES:
    top_supports = [tuple(row) for row in ranked["support"]]
    recheck_rows = label_supports_cuda_bp(top_supports, label_batch_size=LABEL_BATCH_SIZE, dataset_path=None)
    recheck_df = pd.DataFrame(recheck_rows).rename(columns={"min_distance": "actual_min_distance_cuda_bp"})
    ranked_for_merge = ranked.copy()
    ranked_for_merge["support_key"] = ranked_for_merge["support"].apply(tuple)
    recheck_df["support_key"] = recheck_df["indices"].apply(tuple)
    merged = ranked_for_merge.merge(
        recheck_df[["support_key", "actual_min_distance_cuda_bp", "max_zeros", "label_backend"]],
        on="support_key",
        how="left",
    ).drop(columns=["support_key"])
    merged["error_pred_minus_actual"] = merged["predicted_min_distance_cnn"] - merged["actual_min_distance_cuda_bp"]
    display(merged)
else:
    print("Skipped exact CUDA re-check.")

## Outputs

Download these from Kaggle's output panel after the run. With the default parameters they are written under `/kaggle/working/support_point_cnn_runs/<run-name>/`:

- `support_point_cuda_bp_labeled.jsonl` — supervised dataset with exact CUDA-BP labels.
- `support_point_torch_cnn_model.pt` — saved PyTorch CNN model when `TRAINING_BACKEND = "torch"`.
- `support_point_cnn_ridge_model.npz` — saved NumPy ridge model when `TRAINING_BACKEND = "ridge"`.
- `support_point_cnn_predictions.csv` — train/validation predictions, errors, and ±3 flags.

Remember: only rows labeled by `label_backend = cuda-bp-exact` are exact distance evaluations. CNN predictions are for ranking and triage only.